In [ ]:
import json
import numpy as np
import pandas as pd
from rich import print
from matplotlib import pyplot as plt
from matplotlib import rcParams as rc
import dill

rc["font.family"] = "Times New Roman"
rc["font.size"] = 14
rc["figure.figsize"] = (6, 4)
rc["axes.grid"] = True

In [ ]:
istart_channel:int = 3
iend_channel:int = 25
# Load the configuration file
conf = json.load(open("../../atem/data/atem.json"))
times = np.asarray(conf['channels'])[istart_channel:iend_channel] * 1e-6
n_turns = conf['n_turns']

In [ ]:
area:str = "NE"
path:str = f"../../atem/data/11-024_Alberta_{area}.csv"
dheader:list = [f"zoff30[{i}]" for i in range(istart_channel,iend_channel)]
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight", 'pwrline', 'dtm'] + dheader # power line monitor

In [ ]:
raws = pd.read_csv(path)[picker]

In [ ]:
xy = raws[["x_wgs84", "y_wgs84"]].to_numpy()
normalizer = (-1e-9)/ (raws["TranPeak"].values * n_turns).reshape(-1, 1)
raws[[f"zoff30[{i}]" for i in range(istart_channel, iend_channel)]] = raws[[f"zoff30[{i}]" for i in range(istart_channel, iend_channel)]] * normalizer

In [ ]:
line_no = list(raws["Line"].unique())

In [ ]:
# 23~24
istart:int = 0
iend:int = None
index = raws["Line"] == line_no[istart]
print(f"{line_no[istart]=}")

In [ ]:
plt.scatter(xy[:, 0], xy[:, 1], s=1)
plt.scatter(xy[index, 0], xy[index, 1], s=1)
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(f"Line {line_no[istart]}")

In [ ]:
nan_list = raws[raws["y_wgs84"].isna()]["Line"].unique()
print(nan_list)

In [ ]:
counts = []
for line_n in nan_list:
    count = raws[raws["Line"] == line_n]["y_wgs84"].isna().sum()
    counts.append(count)
print(counts)

In [ ]:
plt.scatter(xy[:, 0], xy[:, 1], s=1, c='lightgray', label="All data")
i=0
for line_n in nan_list:
    test_x = raws[raws["Line"] == line_n]["x_wgs84"]
    test_y = raws[raws["Line"] == line_n]["y_wgs84"]
    plt.scatter(test_x, test_y, s=1, c=f"C{i}", label=line_n)
    i += 1
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Line having NaN values in y_wgs84")

In [ ]:
raws.dropna(subset=["y_wgs84"], inplace=True) # Remove rows with NaN values in y_wgs84
raws.fillna(1e-20, inplace=True) # Replace remaining NaN values with a small number (1e-20) to avoid issues in calculations

In [ ]:
dx = 50.
values = []
values_std = []
soundings = []

for i_line, line in enumerate(line_no[istart:iend]):
    df_line = raws[raws['Line']==line]

    # Calculate distance along the "Line"
    xy = df_line[["x_wgs84", "y_wgs84"]].to_numpy()
    distance = np.sqrt(((xy-xy[0,:])**2).sum(axis=1))
    # print(f"Number of NaN values in distance: {np.isnan(distance).sum()}")
    max_distance = distance.max()

    # Determine the no. of soundings per bin.
    if max_distance % dx ==0:
        n_sounding = int(max_distance / dx)
    else:
        n_sounding = int(np.round(max_distance / dx) + 1)

    # Create bins and assign each sounding to a bin
    bins = np.arange(n_sounding) * dx
    df_line.insert(0, 'distance', distance)
    # Bin distances
    df_line['bin'] = pd.cut(df_line['distance'], bins=bins)
    # Compute statistics per bin
    binned = (
        df_line.groupby('bin', observed=False)
            [['distance'] + picker[1:]]
            .mean()
    )
    binned.insert(0, 'Line', line)
    binned_std = (
        df_line.groupby('bin', observed=False)
            [['bheight'] + dheader]
            .std()
    )
    values.append(binned.values)
    values_std.append(binned_std.values)
    soundings.append(n_sounding)

df_data_binned = pd.DataFrame(data=np.vstack(values), columns=['Line', 'distance'] + picker[1:])
df_data_std_binned = pd.DataFrame(data=np.vstack(values_std), columns=['bheight'] + dheader)

In [ ]:
print(f"{soundings=}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from ipywidgets import widgets, interact

In [ ]:
def plot_data(line, x):
    print(line)
    x_line = x[x["Line"]==line]

    fig, axs = plt.subplots(2,1, figsize=(20, 10), constrained_layout=True)
    ax0, ax1 = axs

    # (ax0) Plot the binned locations
    ax0.plot(x["x_wgs84"], x["y_wgs84"], '.')
    ax0.plot(x_line["x_wgs84"], x_line["y_wgs84"], '.')
    ax0.plot(x_line["x_wgs84"].values[0], x_line["y_wgs84"].values[0], 'o')
    ax0.set_aspect(0.2)

    # Normalize the altitude and power line monitor data.
    scaler = MinMaxScaler(feature_range=(x_line['bheight'].min(), x_line['bheight'].max())) # Min-Max scaler for pwrline to be in the same range as bheight
    plm_norm = scaler.fit_transform(x_line['pwrline'].values.reshape([-1,1]))

    # (ax1) Plot the normalized data
    ax2 = ax1.twinx()
    ax1.semilogy(x_line['distance'], np.abs(x_line[dheader]), color='k', lw=2)
    ax2.plot(x_line['distance'], plm_norm, '-', color='red', label='PLM', lw=1.5)
    ax2.plot(x_line['distance'], x_line['bheight'], '-', label='Flight height', lw=1.5, color='magenta')
    # ax1.axvline(2.61e3, c='green', lw = 3) # Infra 01
    # ax1.axvline(8.4e3, c='green', lw = 3) # Altitude dramatically decreasing
    # ax1.axvline(24.68e3, c='green', lw = 3) # Infra 02
    # ax1.axvline(26.5e3, c='green', lw = 3) # Power line (Intersection with road)
    # ax1.axvline(29.4e3, c='green', lw = 3) # Power line (Intersection with road)
    ax2.legend()

In [ ]:
interact(plot_data, line=widgets.Select(options=line_no[istart:iend]), x=widgets.fixed(df_data_binned))

- standard deviation
$$
\sigma=\frac{1}{N}\sqrt{\sum_{i=1^N}{(x_i-\bar{x})^2}}
$$
- field normalized standard deviation (root-mean-squared relative error)
$$
\frac{\sigma}{d}=\frac{1}{N}\sqrt{\sum_{i=1^N}{\frac{(x_i-\bar{x})^2}{x_i^2}}}
$$

In [ ]:
# Extract timeseries data
data_ = df_data_binned[dheader].values.astype(float)
# Extract field normalized standard deviation (root mean squared relative error)
data_rerr = (df_data_std_binned[dheader].values / np.abs(df_data_binned[dheader].values)).astype(float)

In [ ]:
uniqe_lines = np.unique(df_data_binned["Line"].values)

In [ ]:
# Cut-off for bad data (relative error > 3%)
criteria:float = 0.03
channel_id = np.tile(np.arange(data_.shape[1]), (data_.shape[0], 1))
cut_off = (data_rerr>criteria) * (channel_id>=0)

In [ ]:
data_[cut_off] = np.nan
data_rerr[cut_off] = np.nan

In [ ]:
# Plot histogram of relative errors below the cut-off
hi = plt.hist(data_rerr[~cut_off], bins = np.linspace(0,criteria, 100))

In [ ]:
def plot_soundinig(i_sounding):
    plt.errorbar(times, np.abs(data_[i_sounding,:])*1e9, yerr=data_rerr[i_sounding,:], fmt='-o')
    plt.yscale('log')
    plt.xscale('log')
    plt.xlabel("Time (s)")
    plt.ylabel("Field (nT/s)")

interact(plot_soundinig, i_sounding=widgets.IntSlider(min=0, max=data_.shape[0]-1, step=1, value=0))

In [ ]:
filtered_data = pd.DataFrame(
    data = np.hstack(
        (df_data_binned.iloc[:,:8+1].values, data_)
    ),
    columns=df_data_binned.columns.tolist()
)

In [ ]:
Q = interact(plot_data, line=widgets.Select(options=line_no[istart:iend]), x=widgets.fixed(filtered_data))

In [ ]:
criteria_uncertainty:float = 0.05
floors_c:float = 0.05

floors = 5*1e-9/ (df_data_binned["TranPeak"] * n_turns) * floors_c

dobs = df_data_binned[dheader].values.flatten() # Binned Data
std = df_data_std_binned[dheader].values.flatten() # Standard deviation of Binned Data
dobs_std = abs(dobs) * criteria_uncertainty # 0.05 * Binned Data
dobs_std[std > dobs_std] = std[std > dobs_std] # Standard deviation having smaller than 0.05 * dobs will be replaced with 0.05 * dobs.
dobs_std[cut_off.flatten()] = np.inf # Filter out bad data by setting their standard deviation to infinity

dobs_std += np.repeat(floors.values, iend_channel - istart_channel) # Add floors

In [ ]:
print(floors)

In [ ]:
np.sum(1/dobs_std[cut_off.flatten()])

In [ ]:
from simpeg.electromagnetics.utils.em1d_utils import get_vertical_discretization
topography = df_data_binned[['x_wgs84','y_wgs84', 'dtm']].values
source_heights = df_data_binned['bheight'].values
thickness = get_vertical_discretization(21, 2, 1.17)

In [ ]:
from simpeg import maps
import simpeg.electromagnetics.time_domain as tdem
# from pymatsolver import PardisoSolver
from pymatsolver import Solver
import simpeg
from types import SimpleNamespace

In [ ]:
# Define the source waveform. (unit step-off, rectangular, triangular, quarter-sine, custom)
# Triangular waveform. 
start_time = -1.74e-3
peak_time = -0.84e-3
off_time = 0.0
waveform = tdem.sources.TriangularWaveform(
    start_time,
    off_time,
    peak_time
)


In [ ]:
current_times = np.linspace(start_time, off_time)
currents = [waveform.eval(t) for t in current_times]

In [ ]:
radius:float = 5

In [ ]:
input_data_dict = {
    "topography": topography.astype(float),
    "source_heights": source_heights.astype(float),
    "thickness": thickness,
    "time_input_currents":current_times,
    "input_currents":currents,
    "times":times,    
    "data":dobs,
    "data_std":dobs_std,    
}
inp = SimpleNamespace(**input_data_dict)

In [37]:
print(source_location.shape)

(3,)

In [ ]:
source_locations = np.c_[inp.topography[:,0], inp.topography[:,1], inp.topography[:,2]+inp.source_heights]
receiver_locations = np.c_[inp.topography[:,0], inp.topography[:,1],  inp.topography[:,2]+inp.source_heights]
n_sounding = source_locations.shape[0]

source_list = []
receiver_orientation = 'z'
source_orientation = 'z'

for i_sounding in range(n_sounding):    
    # waveform = tdem.sources.PiecewiseLinearWaveform(inp.time_input_currents, inp.input_currents)
    source_location = source_locations[i_sounding, :]
    receiver_location = receiver_locations[i_sounding, :]

    # Receiver list

    dbzdt_receiver = tdem.receivers.PointMagneticFluxTimeDerivative(
            receiver_location, inp.times, "z",
    )

    # Make a list containing all receivers even if just one

    # Must define the transmitter properties and associated receivers

    source_list.append(tdem.sources.CircularLoop(
        [dbzdt_receiver],
        location=source_location,
        waveform=waveform,
        radius=radius,
        i_sounding=i_sounding,
    )
    )

survey = tdem.Survey(source_list)
hz = np.r_[inp.thickness, inp.thickness[-1]]

n_layer = len(hz)
nP = n_sounding * n_layer
sigma_map = maps.ExpMap(nP=nP)

simulation = tdem.Simulation1DLayeredStitched(
    survey=survey, 
    thicknesses=inp.thickness, 
    sigmaMap=sigma_map,
    topo=inp.topography, 
    parallel=True, 
    n_cpu=8, 
    verbose=False, 
    solver=Solver,
)

n_time = inp.times.size

# Create data ojbect
data_object = simpeg.data.Data(survey, dobs=dobs, standard_deviation=inp.data_std)
dmis = simpeg.data_misfit.L2DataMisfit(simulation=simulation, data=data_object)

# nData
inds_active_dobs = dobs.shape[0] - cut_off.sum()
print (f"Percentage of the active data = {inds_active_dobs:,}/{len(dobs):,}={inds_active_dobs.sum()/len(dobs)*100:,.0f}%")

from simpeg.electromagnetics.utils.em1d_utils import set_mesh_1d
import scipy
from discretize import SimplexMesh
from simpeg.regularization.laterally_constrained import LaterallyConstrained

tri = scipy.spatial.Delaunay(inp.topography[:,:2])
mesh_radial = SimplexMesh(tri.points, tri.simplices)
mesh_vertical = set_mesh_1d(hz)
mesh_reg = [mesh_radial, mesh_vertical]

def get_active_edge_indices_with_distance(mesh_radial, mesh_vertical, maximum_distance=1000):
    nz = mesh_vertical.n_cells
    edge_lengths = mesh_radial.edge_lengths
    inds = edge_lengths < maximum_distance
    indActiveEdges = np.tile(inds.reshape([-1,1]), nz).flatten()
    return inds, indActiveEdges

inds, indActiveEdges = get_active_edge_indices_with_distance(
    mesh_radial, mesh_vertical, maximum_distance=500.
)

reg = LaterallyConstrained(
    mesh_reg, 
    mapping=simpeg.maps.IdentityMap(nP=nP),
    alpha_s = 0.,
    alpha_r = 1.,
    alpha_z = 1./2.,
    active_edges=indActiveEdges
)

opt = simpeg.optimization.ProjectedGNCG(maxIter=15, maxIterCG=50)
invProb = simpeg.inverse_problem.BaseInvProblem(dmis, reg, opt)
beta = simpeg.directives.BetaSchedule(coolingFactor=2, coolingRate=1)
betaest = simpeg.directives.BetaEstimate_ByEig(beta0_ratio=1.)
target = simpeg.directives.TargetMisfit(chifact=1)
precond = simpeg.directives.UpdatePreconditioner()
save_model_dict = simpeg.directives.SaveOutputDictEveryIteration()
save_model_dict.outDict = {}

inv = simpeg.inversion.BaseInversion(
    invProb, 
    directiveList=[
        betaest, 
        beta, 
        precond,
        # target, 
        save_model_dict
    ]
)
invProb.counter = opt.counter = simpeg.utils.Counter()
opt.LSshorten = 0.5
opt.remember('xc')
m0 = np.ones(nP) * np.log(1./10.)
mest = inv.run(m0)

In [ ]:
name:str = "./inv_results_atem_full.pik"

dill.dump(save_model_dict.outDict, open(f"{name}", "wb"))

outDict = dill.load(open(f"{name}","rb"))
print(list(outDict.keys()))

In [ ]:
iteration = len(outDict.keys())
# iteration = 6
m = outDict[iteration]['m']
dpred = outDict[iteration]['dpred']
DPRED = dpred.reshape((n_sounding, n_time))
DOBS = dobs.reshape((n_sounding, n_time))
STD = data_object.standard_deviation.reshape((n_sounding, n_time))


In [ ]:
other_values = df_data_binned[['Line', 'distance','x_wgs84','y_wgs84','bheight','pwrline']].values
df_data_binned_pred = pd.DataFrame(data=np.hstack((other_values, -DPRED)), columns=['Line', 'distance','x_wgs84','y_wgs84','bheight','pwrline'] + dheader)

In [ ]:
# i_sounding = 0
def foo(i_sounding):
    plt.plot(times, -DOBS[i_sounding,:], '.')
    plt.plot(times, -DPRED[i_sounding,:])
    plt.yscale('log')
    plt.xscale('log')
    plt.title(f"x: {binned['x_wgs84'].values[i_sounding]:.1f}")
    plt.ylim(1e-12,1e-8)
interact(foo, i_sounding = widgets.IntSlider(min=0, max=n_sounding-1, continous_update=False))    


In [ ]:
from simpeg.electromagnetics.utils.em1d_utils import Stitched1DModel

In [ ]:
line = np.array([int(val.strip('L').strip('T')) for val in df_data_binned['Line'].values])

In [ ]:
model = Stitched1DModel(
    topography=inp.topography,
    physical_property=1./np.exp(m),
    line=line,
    time_stamp=np.arange(n_sounding),
    thicknesses=thickness,    
)

In [ ]:
np.argwhere( uniqe_lines == Q.widget.kwargs['line'])[0][0]

In [ ]:
fig, ax2 = plt.subplots(1,1, figsize=(18, 5))
out, ax2 = model.plot_section(
    x_axis='distance', aspect='auto', dx=dx, alpha=1,
    cmap='turbo',
    clim=(5, 50),
    ax=ax2,
    show_colorbar=False,
    i_line=np.argwhere(
        uniqe_lines == Q.widget.kwargs['line'])[0][0]
        )

cb = plt.colorbar(out, ax=ax2, orientation='horizontal', fraction=0.04)
cb.set_label("Resistivity (ohm-m)")
ax2.set_title(Q.widget.kwargs['line'])

In [ ]:
for line in uniqe_lines:
    fig, axs = plt.subplots(2,1, figsize=(20, 10), constrained_layout=True)
    ax0, ax = axs

    df_data_binned_line = df_data_binned[df_data_binned['Line']==line]
    df_data_binned_pred_line = df_data_binned_pred[df_data_binned_pred['Line']==line]

    ax0.plot(df_data_binned['x_wgs84'], df_data_binned['y_wgs84'], '.')    
    ax0.plot(df_data_binned_line['x_wgs84'], df_data_binned_line['y_wgs84'], '.')
    ax0.plot(df_data_binned_line['x_wgs84'].values[0], df_data_binned_line['y_wgs84'].values[0], 'o')
    ax0.set_aspect(1)
    scaler = MinMaxScaler(feature_range=(df_data_binned_line['bheight'].min(), df_data_binned_line['bheight'].max()))
    plm_norm = scaler.fit_transform(df_data_binned_line['pwrline'].values.reshape([-1,1]))

    ax_1 = ax.twinx()
    ax.semilogy(df_data_binned_line['distance'], -df_data_binned_line[dheader], 'k-', lw=0.5)
    ax.semilogy(df_data_binned_pred_line['distance'], df_data_binned_pred_line[dheader], 'b--', lw=0.5)
    ax_1.plot(df_data_binned_line['distance'], plm_norm, '-', color='red', label='PLM')
    ax_1.plot(df_data_binned_line['distance'], df_data_binned_line['bheight'], '-', label='Flight height')
    ax_1.legend()
    # ax.axvline(2.61e3, c='magenta', lw = 3) # Infra 01
    # ax.axvline(8.4e3, c='magenta', lw = 3) # Altitude dramatically decreasing
    # ax.axvline(24.68e3, c='magenta', lw = 3) # Infra 02
    # ax.axvline(26.5e3, c='magenta', lw = 3) # Power line (Intersection with road)
    # ax.axvline(29.4e3, c='magenta', lw = 3) # Power line (Intersection with road)
    ax.set_xlim(df_data_binned_line['distance'].min(), df_data_binned_line['distance'].max())
    plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (15,5))
ax.semilogy(df_data_binned_line['distance'], -df_data_binned_line[dheader], 'k-', lw=0.7)
ax.semilogy(df_data_binned_pred_line['distance'], df_data_binned_pred_line[dheader], 'b--', lw=0.7)
ax.set_ylim([5e-13,1e-8])
# ax.axvline(2.61e3, c='magenta', lw = 3) # Infra 01
# ax.axvline(8.4e3, c='magenta', lw = 3) # Altitude dramatically decreasing
# ax.axvline(24.68e3, c='magenta', lw = 3) # Infra 02
# ax.axvline(26.5e3, c='magenta', lw = 3) # Power line (Intersection with road)
# ax.axvline(29.4e3, c='magenta', lw = 3) # Power line (Intersection with road)
ax.set_xlim(df_data_binned_line['distance'].min(), df_data_binned_line['distance'].max())
plt.legend(['Observed', 'Predicted'])
plt.tight_layout()

In [ ]:
out = plt.hist(df_data_binned['bheight'].values, bins=100)